# V1-24: 기간 공시 목록 정책과 데이터 계약

## 목적
동일 기업·접수일 기간에서 OpenDART 공시검색 API의 last_reprt_at=N(전체 제출본)과 Y(최종보고서만)을 비교합니다. V1의 기본 정책은 Y이지만, 실제로 무엇이 빠지고 남는지와 전체 페이지를 놓치지 않는지 확인합니다.

공시 목록의 날짜는 **접수일(rcept_dt)**입니다. 재무제표의 사업연도와 혼동하지 않습니다. 이 노트북은 특정 공시의 중요도를 판단하거나 본문을 검색하지 않으며, LLM·LangChain·RAG도 호출하지 않습니다.

확인할 계약: 요청 기업과 날짜에 속한 행만 반환되는가, total_count만큼 모든 페이지를 모았는가, 접수번호가 유일·14자리인가, 원문 뷰어 URL을 구성할 수 있는가, N/Y 결과가 어떻게 다른가.

N과 Y 중 어느 쪽이 일반적으로 더 정확하다는 독립 정답셋은 아직 없습니다. 따라서 N/Y 차이는 **관찰 결과**로만 기록하고 Precision·Recall을 만들어내지 않습니다. 공시검색 [OpenDART 공식 가이드](https://opendart.fss.or.kr/guide/detail.do?apiGrpCd=DS001&apiId=2019001)와 [V1 요구사항](../../docs/v1/REQUIREMENTS.md)을 기준으로 합니다.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import os
import sys
import time

import pandas as pd
from IPython.display import display

In [ ]:
current_dir = Path.cwd().resolve()
if current_dir.name == 'v1':
    project_root = current_dir.parents[1]
elif current_dir.name == 'notebooks':
    project_root = current_dir.parent
else:
    project_root = current_dir

env_path = project_root / '.env'
if not env_path.exists():
    raise FileNotFoundError(f'{env_path} 파일을 먼저 만들어 주세요.')

for raw_line in env_path.read_text(encoding='utf-8').splitlines():
    line = raw_line.strip()
    if line and not line.startswith('#') and '=' in line:
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip(chr(39) + chr(34)))

api_key = os.environ.get('OPENDART_API_KEY')
if not api_key:
    raise ValueError('.env에 OPENDART_API_KEY를 설정해 주세요.')

sys.path.insert(0, str(project_root / 'src'))
from dart.client import _get_bytes, find_corp_info

LIST_URL = 'https://opendart.fss.or.kr/api/list.json'
VIEWER_URL = 'https://dart.fss.or.kr/dsaf001/main.do?rcpNo='
print(f'실행 시각(UTC): {datetime.now(timezone.utc).isoformat()}')
print(f'OpenDART API key loaded: {bool(api_key)} (값은 출력하지 않음)')

In [ ]:
# 같은 기업·접수일 기간에서 N/Y만 바꿉니다. 재무 사업연도 선택 실험이 아닙니다.
CASES = [
    {'id': 'D01', 'company': '삼성전자', 'start': '20240101', 'end': '20241231'},
    {'id': 'D02', 'company': 'SK하이닉스', 'start': '20240101', 'end': '20241231'},
    {'id': 'D03', 'company': '리노공업', 'start': '20240101', 'end': '20241231'},
]
POLICIES = {'N': '전체 제출본', 'Y': '최종보고서만'}
PAGE_COUNT = 100  # 공식 API의 페이지당 최대값
MAX_PAGES = 30     # 초과 시 성공으로 속이지 않고 PAGE_LIMIT으로 중단
display(pd.DataFrame(CASES))

In [ ]:
# 실제 OpenDART 호출 1: 기업명을 공식 corp_code로 확인합니다.
corp_codes = {}
for case in CASES:
    name = case['company']
    if name not in corp_codes:
        info = find_corp_info(name, api_key)
        corp_codes[name] = info['corp_code']
display(pd.DataFrame([{'기업': name, 'corp_code': code} for name, code in corp_codes.items()]))

In [ ]:
# 한 페이지 조회. 오류 013(조회 결과 없음)과 인증·한도·네트워크 오류를 구분합니다.
api_log = []

def request_page(corp_code, start, end, final_only, page_no):
    params = {
        'crtfc_key': api_key, 'corp_code': corp_code,
        'bgn_de': start, 'end_de': end, 'last_reprt_at': final_only,
        'sort': 'date', 'sort_mth': 'desc',
        'page_no': str(page_no), 'page_count': str(PAGE_COUNT),
    }
    started = time.perf_counter()
    try:
        data = json.loads(_get_bytes(LIST_URL, params).decode('utf-8'))
        status_code = data.get('status')
        if status_code == '000':
            status, error = 'OK', None
        elif status_code == '013':
            status, error = 'NO_DATA', None
        else:
            status, error = 'API_ERROR', f"[{status_code}] {data.get('message')}"
    except Exception as exc:
        data, status, error = {}, 'API_ERROR', f'{type(exc).__name__}: {exc}'

    api_log.append({
        'corp_code': corp_code, 'start': start, 'end': end,
        'last_reprt_at': final_only, 'page_no': page_no,
        'status': status, 'latency_ms': (time.perf_counter() - started) * 1000,
        'error': error,
    })
    return {'status': status, 'data': data, 'error': error}

In [ ]:
def fetch_all_pages(corp_code, start, end, final_only):
    first = request_page(corp_code, start, end, final_only, 1)
    if first['status'] == 'NO_DATA':
        return {'status': 'NO_DATA', 'rows': [], 'total_count': 0, 'total_page': 0}
    if first['status'] != 'OK':
        return {'status': 'API_ERROR', 'rows': [], 'error': first['error']}

    data = first['data']
    try:
        total_count = int(data['total_count'])
        total_page = int(data['total_page'])
    except (KeyError, TypeError, ValueError) as exc:
        return {'status': 'INVALID_METADATA', 'rows': [], 'error': str(exc)}
    if total_count < 0 or total_page < 1:
        return {'status': 'INVALID_METADATA', 'rows': [], 'error': 'total_count 또는 total_page가 비정상'}
    if total_page > MAX_PAGES:
        return {'status': 'PAGE_LIMIT', 'rows': [], 'total_count': total_count,
                'total_page': total_page, 'error': f'한도 {MAX_PAGES}페이지 초과'}

    first_rows = data.get('list')
    if not isinstance(first_rows, list) or any(not isinstance(row, dict) for row in first_rows):
        return {'status': 'INVALID_METADATA', 'rows': [], 'error': '첫 페이지 list 형식이 비정상'}
    rows = list(first_rows)
    for page_no in range(2, total_page + 1):
        page = request_page(corp_code, start, end, final_only, page_no)
        if page['status'] != 'OK':
            return {'status': 'PAGE_ERROR', 'rows': [], 'total_count': total_count,
                    'total_page': total_page, 'error': page['error'] or f"{page['status']} on page {page_no}"}
        page_data = page['data']
        try:
            current_count = int(page_data['total_count'])
            current_page_total = int(page_data['total_page'])
            current_page_no = int(page_data['page_no'])
        except (KeyError, TypeError, ValueError) as exc:
            return {'status': 'INVALID_METADATA', 'rows': [], 'total_count': total_count,
                    'total_page': total_page, 'error': f'{page_no}페이지 메타데이터: {exc}'}
        if current_count != total_count or current_page_total != total_page:
            return {'status': 'CHANGED_DURING_PAGING', 'rows': [], 'total_count': total_count,
                    'total_page': total_page, 'error': '페이지 사이에 결과 총수가 바뀜'}
        if current_page_no != page_no:
            return {'status': 'PAGE_ERROR', 'rows': [], 'total_count': total_count,
                    'total_page': total_page, 'error': f'{page_no}페이지 요청에 {current_page_no}페이지 응답'}
        page_rows = page_data.get('list')
        if not isinstance(page_rows, list) or any(not isinstance(row, dict) for row in page_rows):
            return {'status': 'INVALID_METADATA', 'rows': [], 'total_count': total_count,
                    'total_page': total_page, 'error': f'{page_no}페이지 list 형식이 비정상'}
        rows.extend(page_rows)
    receipts = [str(row.get('rcept_no') or '') for row in rows]
    if any(len(number) != 14 or not number.isdigit() for number in receipts):
        return {'status': 'INVALID_IDENTIFIER', 'rows': rows, 'total_count': total_count,
                'total_page': total_page, 'error': '14자리 숫자 접수번호가 아닌 행이 있음'}
    if len(rows) != total_count or len(set(receipts)) != len(rows):
        return {'status': 'INCOMPLETE_RESULT', 'rows': rows, 'total_count': total_count,
                'total_page': total_page, 'error': '수집 행 수·고유 접수번호 수가 API 총수와 다름'}
    return {'status': 'COMPLETE', 'rows': rows, 'total_count': total_count, 'total_page': total_page}

In [ ]:
# 실제 OpenDART 호출 2: 사례별 N/Y를 동일 조건으로 전 페이지 조회합니다.
results = {}
for case in CASES:
    code = corp_codes[case['company']]
    for flag, label in POLICIES.items():
        print(f"[{case['id']}] {case['company']} / {label} / {case['start']}~{case['end']} 시작")
        result = fetch_all_pages(code, case['start'], case['end'], flag)
        results[(case['id'], flag)] = result
        print(f"  {result['status']}: 수집 {len(result['rows'])}건 / API 총수 {result.get('total_count')}건 / {result.get('total_page')}페이지")
        if result.get('error'):
            print(f"  오류: {result['error']}")

display(pd.DataFrame(api_log)[['corp_code', 'last_reprt_at', 'page_no', 'status', 'latency_ms', 'error']])

In [ ]:
# 독립적으로 확인 가능한 응답 계약. 빈 결과를 100% 정확도로 세지 않습니다.
contract_rows = []
for case in CASES:
    for flag in POLICIES:
        result = results[(case['id'], flag)]
        rows = result['rows']
        receipts = [str(row.get('rcept_no') or '') for row in rows]
        valid_dates = sum(
            len(str(row.get('rcept_dt') or '')) == 8 and str(row['rcept_dt']).isdigit()
            and case['start'] <= str(row['rcept_dt']) <= case['end']
            for row in rows
        )
        valid_corps = sum(str(row.get('corp_code') or '') == corp_codes[case['company']] for row in rows)
        valid_receipts = sum(len(number) == 14 and number.isdigit() for number in receipts)
        complete = result['status'] == 'COMPLETE'
        contract_rows.append({
            '사례': case['id'], '기업': case['company'], '정책': flag,
            '상태': result['status'], 'API 총수': result.get('total_count'),
            '수집 행': len(rows), '고유 접수번호': len(set(receipts)),
            '페이지·건수 일치': complete and len(rows) == result['total_count'] == len(set(receipts)),
            '날짜 적합 행': valid_dates, '기업 적합 행': valid_corps,
            '접수번호 형식 적합 행': valid_receipts,
        })

contract_frame = pd.DataFrame(contract_rows)
display(contract_frame)
scored = contract_frame[contract_frame['상태'] == 'COMPLETE']
observed_rows = int(scored['수집 행'].sum())
if observed_rows:
    print(f"Date-filter Accuracy: {int(scored['날짜 적합 행'].sum())}/{observed_rows}")
    print(f"Company-filter Accuracy: {int(scored['기업 적합 행'].sum())}/{observed_rows}")
    print(f"Receipt-ID Format Rate: {int(scored['접수번호 형식 적합 행'].sum())}/{observed_rows}")
else:
    print('평가 가능한 공시 행이 없어 행 단위 정확도를 계산하지 않습니다.')
if len(scored):
    print(f"Page-count Consistency: {int(scored['페이지·건수 일치'].sum())}/{len(scored)}")
else:
    print('Page-count Consistency: 평가 가능한 COMPLETE 사례 없음')
print(f"NO_DATA: {(contract_frame['상태'] == 'NO_DATA').sum()}건 / 오류·페이지 제한: {(~contract_frame['상태'].isin(['COMPLETE', 'NO_DATA'])).sum()}건")

In [ ]:
# 두 정책의 결과 차이를 접수번호 집합으로 확인합니다. 이 차이를 곧바로 Precision/Recall로 해석하지 않습니다.
comparison = []
for case in CASES:
    all_result = results[(case['id'], 'N')]
    final_result = results[(case['id'], 'Y')]
    if all_result['status'] not in {'COMPLETE', 'NO_DATA'} or final_result['status'] not in {'COMPLETE', 'NO_DATA'}:
        comparison.append({'사례': case['id'], '기업': case['company'], '상태': '비교 불가'})
        continue
    all_by_id = {row['rcept_no']: row for row in all_result['rows'] if row.get('rcept_no')}
    final_by_id = {row['rcept_no']: row for row in final_result['rows'] if row.get('rcept_no')}
    only_all = sorted(set(all_by_id) - set(final_by_id))
    only_final = sorted(set(final_by_id) - set(all_by_id))
    comparison.append({
        '사례': case['id'], '기업': case['company'], '상태': '비교 가능',
        'N 전체': len(all_by_id), 'Y 최종': len(final_by_id),
        '공통': len(set(all_by_id) & set(final_by_id)),
        'N에만 있음': len(only_all), 'Y에만 있음': len(only_final),
    })
    print(f"[{case['id']}] N에만 있는 공시 예시")
    display(pd.DataFrame([{
        '접수일': all_by_id[number].get('rcept_dt'),
        '보고서명': all_by_id[number].get('report_nm'),
        '접수번호': number,
    } for number in only_all[:10]]))
    if only_final:
        print('Y에만 있는 공시가 있습니다. API의 필터 범위·갱신 시점을 별도로 확인하세요.')
        display(pd.DataFrame([{
            '접수일': final_by_id[number].get('rcept_dt'),
            '보고서명': final_by_id[number].get('report_nm'),
            '접수번호': number,
        } for number in only_final[:10]]))
display(pd.DataFrame(comparison))

In [ ]:
# 최종보고서 결과의 원문 뷰어 링크 예시. URL 형식을 확인하며 실제 페이지 접근성은 여기서 검사하지 않습니다.
link_examples = []
for case in CASES:
    result = results[(case['id'], 'Y')]
    if result['status'] != 'COMPLETE':
        continue
    for row in result['rows'][:5]:
        number = str(row.get('rcept_no') or '')
        link_examples.append({
            '기업': case['company'], '접수일': row.get('rcept_dt'),
            '보고서명': row.get('report_nm'), '접수번호': number,
            '원문 링크': VIEWER_URL + number if len(number) == 14 and number.isdigit() else None,
        })
display(pd.DataFrame(link_examples))

## 결과를 볼 때

1. 우선 6개 조회의 상태가 COMPLETE인지 봅니다. PAGE_LIMIT·PAGE_ERROR·CHANGED_DURING_PAGING는 누락 가능성이 있어 결과를 성공으로 취급하지 않습니다.
2. 날짜·기업·접수번호·페이지 수집 결과를 확인합니다. 이 값들은 API 응답의 **형식과 필터 계약**을 보는 것이지, 공시 내용의 정확도를 증명하지 않습니다.
3. N/Y의 접수번호 차이와 보고서명을 봅니다. Y를 V1의 기본 정책으로 두는 요구사항은 이미 있지만, 실제 차이와 이용자에게 생길 제한은 이 출력으로 검토합니다.
4. 2024년 접수일 기간의 목록은 2024 **사업연도 보고서 목록**과 같지 않습니다. 이후 사업보고서 원문 RAG는 별도의 문서 선택·추출·검색 평가가 필요합니다.
5. 결과가 저장된 뒤에만 해석·보고서를 작성합니다. 이 노트북에서는 공시 중요도, 목록 Precision/Recall, 링크 실제 열림 여부를 평가했다고 주장하지 않습니다.